# Get current price data to test the model on real stocks

In [1]:
import numpy as np
import yfinance as yf
from sklearn.covariance import LedoitWolf
from scipy.stats import gmean
import pandas as pd

Selecting 8 indices and calculating their covariance using the LedoitWolf method

In [2]:
start = '2023-09-24'
end = '2026-03-04'
tickers = ['^FTSE', '^GSPC', '^DJI', '^IXIC', '^N225', '^HSI', '^STOXX50E', '^FCHI']

def create_cov(tickers, start, end):
    # download the data
    prices = yf.download(tickers, start, end, auto_adjust=False)['Adj Close']

    # 1. Prepare your data (dropp_puting NAs is crucial for the solver)
    log_returns = np.log(prices / prices.shift(1)).dropna()

    # 2. Initialize the Ledoit-Wolf estimator
    lw = LedoitWolf()

    # 3. Fit the model to your returns 
    # Note: sklearn expects (observations, features), which matches your log_returns layout
    lw.fit(log_returns)

    # 4. Extract the shrunKcovariance matrix
    # This is the "cleaner" version of your cov_matrix
    shrunk_cov = lw.covariance_

    # 5. Annualize (same as your original code)
    shrunk_cov_annual = shrunk_cov * 252

    prices.to_csv("data/etf_prices_04032026.csv")
    pd.DataFrame(shrunk_cov_annual, columns=tickers, index=tickers).to_csv("data/etf_covariance_matrix_04032026.csv")



etf_shrunk_cov_annual = np.array(pd.read_csv("data/etf_covariance_matrix_04032026.csv", index_col=0))
etf_prices = pd.read_csv("data/etf_prices_04032026.csv", index_col='Date')

In [3]:
# get current prices
S0_etf = etf_prices.iloc[-1, :]
S0_etf = np.array(S0_etf)

# get geometric average of the stocKprices
S0_etf_geo_mean = gmean(S0_etf)

# choose strike price
K_etf = 1.01 * S0_etf_geo_mean

print(f'Strike: {K_etf}')

Strike: 16588.014103785066


Get the dividend yields for each stock

Create parameter dictionary

In [4]:
r = 0.0375             # current BOE rate
div = 0                # no dividends on ETFs
T = 0.25               # exercise time
d = len(tickers)       # number of assets

p_call = {}
p_call['strike'] = K_etf
p_call['rate'] = r     
p_call['dividend'] = div
p_call['expiration'] = T
p_call['dim'] = d
p_call['S0'] = S0_etf
p_call['volatility'] = None
p_call['correlation'] = None
p_call['covariance'] = etf_shrunk_cov_annual
p_call['numTimeStep'] = 50
p_call['callput'] = 'call'

p_put = p_call.copy()
p_put['callput'] = 'put'

In [5]:
# Import the model
import glsm_geobasketcall

Run the G-LSM model to estimate the price of such an option

In [6]:
# Call the main function
glsm_call_etf, _ = glsm_geobasketcall.main(p_call)

Number of basis functions: 361
---------------------------------------------
run trial no.1, price = 301.5516
---------------------------------------------
run trial no.2, price = 309.3087
---------------------------------------------
run trial no.3, price = 298.9324
---------------------------------------------
run trial no.4, price = 299.3506
---------------------------------------------
run trial no.5, price = 303.0936
---------------------------------------------
run trial no.6, price = 309.7918
---------------------------------------------
run trial no.7, price = 308.5876
---------------------------------------------
run trial no.8, price = 297.4359
---------------------------------------------
run trial no.9, price = 307.4362
---------------------------------------------
run trial no.10, price = 311.2642
---------------------------------------------

Mean price: 304.6753
Std dev:    4.8989


Now price the call, we checKthe number with the put call parity

In [7]:
# Call the main function
glsm_put_etf, _ = glsm_geobasketcall.main(p_put)

Number of basis functions: 361
---------------------------------------------
run trial no.1, price = 372.2712
---------------------------------------------
run trial no.2, price = 376.3403
---------------------------------------------
run trial no.3, price = 368.3862
---------------------------------------------
run trial no.4, price = 374.4317
---------------------------------------------
run trial no.5, price = 370.5735
---------------------------------------------
run trial no.6, price = 367.6255
---------------------------------------------
run trial no.7, price = 371.1606
---------------------------------------------
run trial no.8, price = 372.1932
---------------------------------------------
run trial no.9, price = 375.8221
---------------------------------------------
run trial no.10, price = 372.6224
---------------------------------------------

Mean price: 372.1427
Std dev:    2.7311


### Risk-Neutral Pricing of European Geometric Mean Options

We assume that, under the risk-neutral probability measure:
$$dS_i(t) = S_i(t)((r-q_i)dt + \sigma_i dW_i(t)), \quad \text{for } i=1, \dots, n$$

Where:
* **$r$** is the interest rate.
* **$\sigma_i$** is the volatility.
* **q_i** is the continous dividend yield of asset $i$
* **$\{W_i(t), t \geq 0\}$** is a standard Brownian motion such that $d\langle W_i, W_j \rangle(t) = \rho_{i,j}dt$ for $i \neq j$.

Note that it is assumed that there is no dividend yield here, the analysis does not work if there is a divident. I will address that later.

Then, the geometric mean is given by:
$$\left(\prod_{i=1}^{n} S_i(T)\right)^{\frac{1}{n}} = \left(\prod_{i=1}^{n} S_i(0)\right)^{\frac{1}{n}} \exp\left( \left(r - \bar{q} -\frac{1}{2n} \sum_{i=1}^{n} \sigma_i^2\right)T + \frac{1}{n} \sum_{i=1}^{n} \sigma_i W_i(T) \right),$$
where $\bar{q} = \frac{1}{n}\sum_{i=1}^{n}q_i$ is the average yield.

Let the aggregate volatility $\sigma$ and the standard normal variable $\xi$ be defined as:
$$\sigma = \sqrt{\frac{1}{n^2} \left( \sum_{i=1}^{n} \sigma_i^2 + 2\sum_{i < j} \rho_{i,j} \sigma_i \sigma_j \right)}, \quad \xi = \frac{\frac{1}{n} \sum_{i=1}^{n} \sigma_i W_i(T)}{\sigma \sqrt{T}}$$

Since we have a covariance matrix ($\Sigma$) instead of volatility ($\sigma_i$) and correlations ($\rho_{i,j}$), we can use the elemnts of the covariance matrix are defined as $\Sigma_{i,j} = \sum_{i,j}\sigma_i\sigma_j\phi_{i,j}$ to rewrite the equation for $\sigma$ as

$$\sigma = \sqrt{\frac{1}{n^2}\sum_{i=1}^{n}\sum_{j=1}^{n}\Sigma_{i,j}}.$$

Then, $\xi \sim N(0,1)$. Let:
$$F = \left(\prod_{i=1}^{n} S_i(0)\right)^{\frac{1}{n}} \exp\left( \left(r - \bar{q} -\frac{1}{2n} \sum_{i=1}^{n} \sigma_i^2 + \frac{1}{2}\sigma^2\right)T \right).$$
The quantity $F$ represents the risk-neutral expected value of the geometric basket at maturity. When dividends are present, the forward drift of the geometric basket depends on the average dividend yield $\bar{q}$.

The geometric mean at time $T$ can be expressed as:
$$\left(\prod_{i=1}^{n} S_i(T)\right)^{\frac{1}{n}} = F e^{-\frac{1}{2}\sigma^2 T + \sigma \sqrt{T} \xi}$$

Following the derivation of the **Black-Scholes formula**, the expected payoff is:
$$E\left[\max\left(\left(\prod_{i=1}^{n} S_i(T)\right)^{\frac{1}{n}} - K, 0\right)\right] = F\Phi(d_1) - K\Phi(d_2)$$

Where $\Phi$ is the standard normal CDF, and:
$$d_1 = \frac{\ln(F/K) + \frac{1}{2}\sigma^2 T}{\sigma \sqrt{T}}, \quad d_2 = d_1 - \sigma \sqrt{T}$$

The option value is then:
$$C = e^{-rT} [F\Phi(d_1) - K\Phi(d_2)].$$

We can then use the put-call parity to derive the price of the put option

Put-call partiy:
$$C - P = e^{-rT}(F-K)$$

Rearranging for P, and substituting in C from above we find

\begin{align}
P &= e^{-rT} [F\Phi(d_1) - K\Phi(d_2)] - e^{-rT}(F-K)\\
P &= e^{-rT}[K(1-\Phi(d_2)) - F(1-\Phi(d_1))].
\end{align}

We now use the fact that $1-\Phi(a) = \Phi(-a)$ to find the value of the put option is

$$P=e^{-rT}[K\Phi(-d_2) - F\Phi(-d_1)].$$

The expression above provides a closed-form benchmark for the European geometric basket call and put. For Bermudan (or American) exercise, no closed form is available in general because the problem becomes an optimal stopping problem in a multidimensional state space. The Bermudan price is at least as large as the European price, since it includes the additional right to exercise early. In numerical experiments, we therefore expect Bermudan prices to lie above the European benchmark, with the gap reflecting the value of early exercise.

In [8]:
# used to find the cumulative distribution values
from scipy.stats import norm

# calculating the price of the call and the put with put call parity

def geo_basket_bs_prices(Sigma_annual, S0_vec, K, r, T, dividend_array=None):
    """
    European call/put price for a geometric basket option under multivariate Black-Scholes.

    Sigma_annual: (d,d) covariance matrix of log-returns (annualized)
    S0_vec: (d,) spot prices
    K: strike (same units as S0)
    r: risk-free rate
    T: maturity in years
    dividend_array: (d,) dividend yields q_i (annualized), or None/0 for no dividends
    """
    S0_vec = np.asarray(S0_vec, dtype=float)
    d = S0_vec.size

    # geometric mean spot
    G0 = np.exp(np.mean(np.log(S0_vec)))

    # effective variance of log geometric mean
    ones = np.ones(d)
    sigmaG2 = (ones @ Sigma_annual @ ones) / (d**2)
    sigmaG = np.sqrt(sigmaG2)

    # average dividend yield
    if dividend_array is None:
        qbar = 0.0
    else:
        qbar = float(np.mean(np.asarray(dividend_array, dtype=float)))

    # average marginal variances
    sig_i2 = np.diag(Sigma_annual)
    mean_sig_i2 = float(np.mean(sig_i2))

    # effective dividend yield for the geometric basket process
    q_eff = qbar + 0.5 * mean_sig_i2 - 0.5 * sigmaG2

    # forward mean under Q
    F = G0 * np.exp((r - q_eff) * T)

    if sigmaG < 1e-14:
        # degenerate case: almost deterministic
        C = np.exp(-r*T) * max(F - K, 0.0)
        P = np.exp(-r*T) * max(K - F, 0.0)
        return C, P, q_eff, sigmaG

    d1 = (np.log(F / K) + 0.5 * sigmaG2 * T) / (sigmaG * np.sqrt(T))
    d2 = d1 - sigmaG * np.sqrt(T)

    C = np.exp(-r*T) * (F * norm.cdf(d1) - K * norm.cdf(d2))
    P = np.exp(-r*T) * (K * norm.cdf(-d2) - F * norm.cdf(-d1))
    return C, P, q_eff, sigmaG




In [9]:
C, P, _, _ = geo_basket_bs_prices(etf_shrunk_cov_annual, S0_etf, K_etf, r, T)

print(f'Call price: {C}')
print(f'Put price: {P}')

Call price: 307.2259186925064
Put price: 359.14334164802307


# Error in G-LSM method

In [10]:
glsm_call_etf = 304.9931
glsm_put_etf =  372.3066


error_call = abs(C - glsm_call_etf) / abs(C)
error_put = abs(P - glsm_put_etf) / abs(P)

print(f'G-LSM call price: {glsm_call_etf:.2}')
print(f'G-LSM put price: {glsm_put_etf:.2}')

print(f'Call error: {error_call*100:.2}%')
print(f'Put error: {error_put*100:.2}%')

G-LSM call price: 3e+02
G-LSM put price: 3.7e+02
Call error: 0.73%
Put error: 3.7%


# Pricing options with tree model

We use the following binomial tree model to price the put options, since the BS model can only give us a lower bound.

In [11]:
def bermudan_geo_basket_tree(p, Sigma_annual):
    """
    Bermudan option benchmark via 1D binomial tree on the geometric mean basket,
    with parameters matched to the multivariate BS geometric basket dynamics.
    """
    d = p['dim']
    S0_vec = np.asarray(p['S0'], dtype=float)
    G0 = np.exp(np.mean(np.log(S0_vec)))

    # effective variance of log geometric mean
    ones = np.ones(d)
    sigmaG2 = (ones @ Sigma_annual @ ones) / (d**2)
    sigmaG = np.sqrt(sigmaG2)

    # average dividend yield on components
    div_arr = p.get('dividend_array', None)
    if div_arr is None:
        qbar = 0.0
    else:
        qbar = float(np.mean(np.asarray(div_arr, dtype=float)))

    # average marginal variances
    mean_sig_i2 = float(np.mean(np.diag(Sigma_annual)))

    # effective dividend yield for geometric basket
    q_eff = qbar + 0.5 * mean_sig_i2 - 0.5 * sigmaG2

    # tree params
    N = int(p['numTimeStep'])
    T = float(p['expiration'])
    r = float(p['rate'])
    K = float(p['strike'])
    dt = T / N

    u = np.exp(sigmaG * np.sqrt(dt))
    d_down = 1.0 / u

    # risk-neutral up prob for GBM with carry (r - q_eff)
    p_up = (np.exp((r - q_eff) * dt) - d_down) / (u - d_down)
    df = np.exp(-r * dt)

    # terminal node prices
    S = G0 * (u ** np.arange(N, -N - 1, -2))

    if p['callput'] == 'call':
        V = np.maximum(S - K, 0.0)
    else:
        V = np.maximum(K - S, 0.0)

    # backward induction with Bermudan exercise at every step
    for i in range(N - 1, -1, -1):
        V = df * (p_up * V[:-1] + (1 - p_up) * V[1:])
        S = G0 * (u ** np.arange(i, -i - 1, -2))

        if p['callput'] == 'call':
            EV = np.maximum(S - K, 0.0)
        else:
            EV = np.maximum(K - S, 0.0)

        V = np.maximum(V, EV)

    return float(V[0])

Selecting 8 stocks in the sp500 and calculating their covariance using the LedoitWolf method

In [12]:
start = '2023-09-24'
end = '2026-03-04'
tickers = ['AAPL', 'TSLA', 'LLY', 'BRK-B', 'V', 'BAC', 'WFC', 'IBM']

def create_cov(tickers, start, end):
    # download the data
    prices = yf.download(tickers, start, end, auto_adjust=False)['Adj Close']

    # 1. Prepare your data (dropping NAs is crucial for the solver)
    log_returns = np.log(prices / prices.shift(1)).dropna()

    # 2. Initialize the Ledoit-Wolf estimator
    lw = LedoitWolf()

    # 3. Fit the model to your returns 
    # Note: sklearn expects (observations, features), which matches your log_returns layout
    lw.fit(log_returns)

    # 4. Extract the shrunKcovariance matrix
    # This is the "cleaner" version of your cov_matrix
    shrunk_cov = lw.covariance_

    # 5. Annualize (same as your original code)
    shrunk_cov_annual = shrunk_cov * 252

    prices.to_csv("data/stock_prices_04032026.csv")
    pd.DataFrame(shrunk_cov_annual, columns=tickers, index=tickers).to_csv("data/covariance_matrix_04032026.csv")


shrunk_cov_annual = np.array(pd.read_csv("data/covariance_matrix_04032026.csv", index_col=0))
prices = pd.read_csv("data/stock_prices_04032026.csv", index_col='Date')

In [13]:
# get current prices
S0 = prices.iloc[-1, :]
S0 = np.array(S0)

# get geometric average of the stocKprices
S0_geo_mean = gmean(S0)

# choose strike price
K = 32.5/40 * S0_geo_mean

print(f'Strike: {K}')

Strike: 204.79849399273994


Get the dividend yields for each stock

In [14]:
# yields = []
# for t in tickers:
#     info = yf.Ticker(t).info
#     # Use trailingAnnualDividendYield or dividendYield
#     y = info.get('dividendYield', 0) 
#     yields.append(y if y is not None else 0)

# yields[-1] = 2.68 # for some reason yfinance didn't get the right divident yield for IBM...
# yields

yields_04032026 = [0.39, 0, 0.62, 0, 0.84, 2.24, 2.18, 2.68]

Create parameter dictionary

In [15]:
r = 0.0375             # current BOE rate
div = np.mean(yields_04032026)  # arithmetic mean of dividend yield
T = 0.25               # exercise time
d = len(tickers)       # number of assets

p_call_d = {}
p_call_d['strike'] = K
p_call_d['rate'] = r     
p_call_d['dividend'] = div
p_call_d['dividend_array'] = yields_04032026
p_call_d['expiration'] = T
p_call_d['dim'] = d
p_call_d['S0'] = S0
p_call_d['volatility'] = None
p_call_d['correlation'] = None
p_call_d['covariance'] = shrunk_cov_annual
p_call_d['numTimeStep'] = 50
p_call_d['callput'] = 'call'

p_put_d = p_call_d.copy()
p_put_d['callput'] = 'put'

In [16]:
# Call the main function
glsm_call_d, _ = glsm_geobasketcall.main(p_call_d)

Number of basis functions: 361
---------------------------------------------
run trial no.1, price = 45.8281
---------------------------------------------
run trial no.2, price = 45.8287
---------------------------------------------
run trial no.3, price = 45.8566
---------------------------------------------
run trial no.4, price = 45.8044
---------------------------------------------
run trial no.5, price = 45.8456
---------------------------------------------
run trial no.6, price = 45.8375
---------------------------------------------
run trial no.7, price = 45.9083
---------------------------------------------
run trial no.8, price = 45.8372
---------------------------------------------
run trial no.9, price = 45.8959
---------------------------------------------
run trial no.10, price = 45.8765
---------------------------------------------

Mean price: 45.8519
Std dev:    0.0309


In [17]:
# Call the main function
glsm_put_d, _ = glsm_geobasketcall.main(p_put_d)

Number of basis functions: 361
---------------------------------------------
run trial no.1, price = 16.2225
---------------------------------------------
run trial no.2, price = 16.3412
---------------------------------------------
run trial no.3, price = 16.2852
---------------------------------------------
run trial no.4, price = 16.2190
---------------------------------------------
run trial no.5, price = 16.0708
---------------------------------------------
run trial no.6, price = 16.6165
---------------------------------------------
run trial no.7, price = 16.3126
---------------------------------------------
run trial no.8, price = 15.9553
---------------------------------------------
run trial no.9, price = 16.2673
---------------------------------------------
run trial no.10, price = 16.3434
---------------------------------------------

Mean price: 16.2634
Std dev:    0.1660


In [18]:
# accurate estimate with binomial tree model
benchmark_call = bermudan_geo_basket_tree(p_call_d, shrunk_cov_annual)
benchmark_put = bermudan_geo_basket_tree(p_put_d, shrunk_cov_annual)

print(f'Benchmark call: {benchmark_call}')
print(f'Benchmark put: {benchmark_put}')


Benchmark call: 47.26119092140152
Benchmark put: 15.895866079491455


In [19]:
error_call_d = abs(benchmark_call - glsm_call_d) / abs(benchmark_call)
error_put_d = abs(benchmark_put - glsm_put_d) / abs(benchmark_put)

print(f'G-LSM call price: {glsm_call_d:.2}')
print(f'G-LSM put price: {glsm_put_d:.2}')

print(f'Call error: {error_call_d*100:.2}%')
print(f'Put error: {error_put_d*100:.2}%')

G-LSM call price: 4.6e+01
G-LSM put price: 1.6e+01
Call error: 3.0%
Put error: 2.3%


# Test the accuracy of the model vs moneyness

Preliminary testing suggests that this model may have larger errors when the strike price is far away from the average stock price. We will use the no dividend data to test the accuracy depending on the moneyness of the option, i.e. how far away the strike price is from the average initial stock price.

Testing was done on the university server and results are shown in "test_results.ipynb"